In [3]:
cd ~/Desktop/Новая_папка/

/Users/annatezelnikova/Desktop/Новая_папка


In [5]:
%%writefile Dockerfile

FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir fastapi uvicorn langchain-openai python-dotenv

EXPOSE 8000

ENV RUN_AGENTS=true
ENV OUTPUT_DIR=demo

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [13]:
%%writefile Dockerfile

FROM python:3.11-slim
WORKDIR /app
COPY requirements-server.txt .
RUN pip install --no-cache-dir -r requirements-server.txt
COPY server.py .
ENV OUTPUT_DIR=/app/static
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [1]:
%%writefile llm_client.py

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_HOST = os.getenv('OPENAI_API_HOST')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
MODEL_NAME = "openai/gpt-oss-120b"

from dataclasses import dataclass
from langchain_openai import ChatOpenAI

@dataclass
class LLMClient:
    model_name: str = MODEL_NAME
    base_url: str = OPENAI_API_HOST
    api_key: str = OPENAI_API_KEY

    def __post_init__(self):
        self._client = ChatOpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            model=self.model_name,
            timeout=None,
        )

    def generate(self, prompt: str) -> str:
        response = self._client.invoke(prompt)
        return response.content

Overwriting llm_client.py


In [1]:
ls 

agents.py         llm_client.py     requirements.txt
Dockerfile        main.py           Untitled3.ipynb


In [11]:
ls -a

./                  .ipynb_checkpoints/ llm_client.py       Untitled3.ipynb
../                 agents.py           main.py
.env                Dockerfile          requirements.txt


In [ ]:
base_url="http://146.103.115.38:11435/v1"
api_key="sk-RG81OLqjbmLB9fN7NiT6p75KiGfddAi4"
model="openai/gpt-oss-120b"

In [3]:
%%writefile .env

OPENAI_API_HOST=http://146.103.115.38:11435/v1
OPENAI_API_KEY=sk-RG81OLqjbmLB9fN7NiT6p75KiGfddAi4

Overwriting .env


In [13]:
cat .env


OPENAI_API_HOST=http://146.103.115.38:11435/v1
OPENAI_API_KEY=sk-RG81OLqjbmLB9fN7NiT6p75KiGfddAi4


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_HOST = os.getenv('OPENAI_API_HOST')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
MODEL_NAME = "openai/gpt-oss-120b"

In [7]:
OPENAI_API_HOST

'http://146.103.115.38:11435/v1'

In [8]:
OPENAI_API_KEY

'sk-RG81OLqjbmLB9fN7NiT6p75KiGfddAi4'

In [12]:
from langchain_openai import ChatOpenAI
client = ChatOpenAI(
    base_url=OPENAI_API_HOST,
    api_key=OPENAI_API_KEY,
    model=MODEL_NAME,
    timeout=None,
)
client.invoke('hi')

/Applications/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 66, 'total_tokens': 98, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-87a2ef54ba00a50b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c994d-2069-7593-970d-f7ea2751ed76-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 32, 'total_tokens': 98, 'input_token_details': {}, 'output_token_details': {}})

In [22]:
%%writefile agents.py

from dataclasses import dataclass
from pathlib import Path
from abc import ABC, abstractmethod
import os, shutil

# ===== Контекст между агентами =====
@dataclass
class AgentContext:
    requirements: str
    plan: str | None = None
    domain: str | None = None
    tests: str | None = None
    code: str | None = None
    review: str | None = None

# ===== Базовый класс агента =====
class BaseAgent(ABC):
    def __init__(self, llm: 'LLMClient'):
        self.llm = llm

    @abstractmethod
    def run(self, ctx: AgentContext) -> AgentContext:
        ...

# ===== Планирование =====
class PlannerAgent(BaseAgent):
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = (
            "Разбей требования на этапы: DDD → Tests → Code → Review → Build.\n"
            f"Требования: {ctx.requirements}"
        )
        ctx.plan = self.llm.generate(prompt)
        return ctx

# ===== Анализ доменной модели =====
class DDDAnalystAgent(BaseAgent):
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"Опиши доменную модель по требованиям:\n{ctx.requirements}"
        ctx.domain = self.llm.generate(prompt)
        return ctx

# ===== Генерация тестов =====
class TestEngineerAgent(BaseAgent):
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"Сгенерируй pytest-тесты для доменной модели:\n{ctx.domain}"
        ctx.tests = self.llm.generate(prompt)
        return ctx

# ===== Разработчик (генерация файлов) =====
class DeveloperAgent(BaseAgent):
    def run(self, ctx: AgentContext) -> AgentContext:
        from pathlib import Path
        import shutil
        import os

        output_dir = Path(os.getenv("OUTPUT_DIR", "demo"))

        # Удаляем старую папку
        shutil.rmtree(output_dir, ignore_errors=True)
        output_dir.mkdir(parents=True, exist_ok=True)

        # Промпты
        html_prompt = f"""
        Generate clean modern HTML page.
        Only return pure HTML.
        Requirements:
        {ctx.requirements}
        """

        css_prompt = """
        Generate clean modern CSS.
        Only return pure CSS.
        """

        js_prompt = """
        Generate clean JavaScript.
        Only return pure JS.
        """

        # Генерация
        index_html = self.llm.generate(html_prompt)
        styles_css = self.llm.generate(css_prompt)
        script_js = self.llm.generate(js_prompt)

        # Добавляем UTF-8
        if not styles_css.strip().startswith("@charset"):
            styles_css = '@charset "UTF-8";\n\n' + styles_css

        # Сохраняем файлы
        (output_dir / "index.html").write_text(index_html, encoding="utf-8")
        (output_dir / "styles.css").write_text(styles_css, encoding="utf-8")
        (output_dir / "script.js").write_text(script_js, encoding="utf-8")

        return ctx

# ===== Code Review =====
class ReviewerAgent(BaseAgent):
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = (
            "Проведи code review сгенерированных файлов и верни список улучшений."
        )
        ctx.review = self.llm.generate(prompt)
        return ctx

# ===== Завершение =====
class BuilderAgent(BaseAgent):
    def run(self, ctx: AgentContext) -> AgentContext:
        # Можно добавить финальные действия, если нужно
        print("Генерация статических файлов завершена.")
        return ctx

Overwriting agents.py


In [21]:
%%writefile main.py

import os
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles

from agents import (
    AgentContext,
    PlannerAgent,
    DDDAnalystAgent,
    TestEngineerAgent,
    DeveloperAgent,
    ReviewerAgent,
    BuilderAgent
)
from llm_client import LLMClient


# ==========================================
# 1. Запуск мультиагентной системы
# ==========================================

def run_multi_agent_pipeline(requirements: str):

    print("▶ Запуск мультиагентной системы...\n")

    ctx = AgentContext(requirements=requirements)
    llm = LLMClient()

    agents = [
        PlannerAgent(llm),
        DDDAnalystAgent(llm),
        TestEngineerAgent(llm),
        DeveloperAgent(llm),   # ← генерирует HTML/CSS/JS
        ReviewerAgent(llm),
        BuilderAgent(llm),
    ]
    

    for agent in agents:
        print(f"▶ {agent.__class__.__name__}")
        ctx = agent.run(ctx)

    print("\n Генерация завершена!\n")
    return ctx


# ==========================================
# 2. Генерация файлов при старте контейнера
# ==========================================

if os.getenv("RUN_AGENTS", "true") == "true":

    requirements = os.getenv(
        "APP_REQUIREMENTS",
        "Создай одностраничный сайт с заголовком и кнопкой."
    )

    run_multi_agent_pipeline(requirements)


# ==========================================
# 3. FastAPI сервер для отдачи demo/
# ==========================================

output_dir = os.getenv("OUTPUT_DIR", "demo")

app = FastAPI(title="Multi-Agent Generated Web App")

app.mount(
    "/",
    StaticFiles(directory=output_dir, html=True),
    name="static"
)


@app.get("/health")
def health():
    return {"status": "ok"}


print(" Приложение будет доступно по адресу:")
print(" http://localhost:8000")

Overwriting main.py


In [26]:
import subprocess
result = subprocess.run(["docker", "compose", "up", "--build"], capture_output=True, text=True)
print(result.stdout)

In [20]:
import subprocess
result = subprocess.run(["ls", "-l"], capture_output=True, text=True)
print(result.stdout)

total 80
drwxr-xr-x  3 annatezelnikova  staff     96 26 февр. 14:33 __pycache__
-rw-r--r--@ 1 annatezelnikova  staff   3441 19 февр. 17:18 agents.py
-rw-r--r--@ 1 annatezelnikova  staff    121 19 февр. 15:23 Dockerfile
-rw-r--r--@ 1 annatezelnikova  staff    722 19 февр. 14:13 llm_client.py
-rw-r--r--@ 1 annatezelnikova  staff   2793 26 февр. 14:33 main.py
-rw-r--r--@ 1 annatezelnikova  staff     47 19 февр. 15:36 requirements.txt
-rw-r--r--  1 annatezelnikova  staff  17983 26 февр. 14:46 Untitled3.ipynb



In [29]:
!ls

__pycache__        Dockerfile         repo
agents.py          llm_client.py      requirements.txt
docker-compose.yml main.py            Untitled3.ipynb


In [27]:
!docker compose up --build -d

empty compose file


In [28]:
!mkdir repo

In [32]:
1. repo с app.py, Dockerfile, docker-compose.yml
2. выполнить команду `docker compose up --build -d` с помощью subprocess
3. вернуть ссылку на веб-приложение

SyntaxError: invalid syntax (2775188750.py, line 1)

In [33]:
!ls

__pycache__        Dockerfile         repo
agents.py          llm_client.py      requirements.txt
docker-compose.yml main.py            Untitled3.ipynb


In [97]:
from time import time
import subprocess

from langchain.tools import tool

@tool
def run_docker() -> str:
    """
    Run application by docker compose command
    
    """
    path = 'repo/docker-compose.yml'
    command = f"docker compose -f {path} up --build -d".split()
    result = subprocess.run(command, capture_output=True, text=True)
    return result

@tool
def get_current_time() -> str:
    """
    Return current time
    """
    return str(time())

In [ ]:
base_url="https://api.deepinfra.com/v1/openai"
api_key="vwSihJMehnkZZ7bMZwluDy4EaeQNKN3x"
model="openai/gpt-oss-120b"

In [112]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

base_url="https://api.deepinfra.com/v1/openai"
api_key="vwSihJMehnkZZ7bMZwluDy4EaeQNKN3x"
model="openai/gpt-oss-120b"

model = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model=model
)
agent = create_agent(
    model,
    tools=[run_docker, get_current_time],
    system_prompt="You are a helpfull assistant."
)

result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "what time is it?"}
    ]
    },
)
result

{'messages': [HumanMessage(content='what time is it?', additional_kwargs={}, response_metadata={}, id='b43521b6-ea3c-4117-b3aa-b21513cf1c89'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 137, 'total_tokens': 178, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'estimated_cost': 1.3133000000000001e-05}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-RvrdIb7NMGeihgDxKW00AOwL', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9a0a-9cd0-7003-8479-8c6f0d005454-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'chatcmpl-tool-33569812a667498fa4cdf18d029cea11', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 41, 'total_tokens': 178, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content='1772110914.409638', name='get_c

In [114]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

base_url="http://146.103.115.38:11435/v1"
api_key="sk-RG81OLqjbmLB9fN7NiT6p75KiGfddAi4"
model="openai/gpt-oss-120b"

model = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model=model
)
agent = create_agent(
    model,
    tools=[run_docker, get_current_time],
    system_prompt="You are a helpfull assistant."
)

result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "what time is it?"}
    ]
    },
)
result

{'messages': [HumanMessage(content='what time is it?', additional_kwargs={}, response_metadata={}, id='38673d1b-5df2-4c56-866d-2cc687ad9d3f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 149, 'total_tokens': 195, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-b5f12b0e8d1e2135', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9a36-07b7-7420-89ab-cc4d26a57a00-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'chatcmpl-tool-b04d2cdf0f8f644c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 46, 'total_tokens': 195, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content='1772113758.803573', name='get_current_time', id='37e14537-1143-494d-b01e-b8fad73e6fbc', tool_call

In [55]:
!touch repo/docker-compose.yml

In [54]:
!cd repo && ls 

tmp.txt


In [58]:
!ls repo

docker-compose.yml


In [59]:
!git clone https://github.com/tezzanna/demo.git

Cloning into 'demo'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 14 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 4.62 KiB | 4.62 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [60]:
!ls

__pycache__        docker-compose.yml main.py            Untitled3.ipynb
agents.py          Dockerfile         repo
demo               llm_client.py      requirements.txt


In [63]:
!cp demo/* repo/

In [103]:
!cd repo && docker compose down -v

[+] down 0/1
 ⠋ Container repo-web-1 Stopping                                            0.1s
[+] down 0/1
 ⠙ Container repo-web-1 Stopping                                            0.2s
[+] down 1/2
 ✔ Container repo-web-1 Removed                                             0.2s
 ⠋ Network repo_default Removing                                            0.1s
[+] down 1/2
 ✔ Container repo-web-1 Removed                                             0.2s
 ⠙ Network repo_default Removing                                            0.1s
[+] down 2/2
 ✔ Container repo-web-1 Removed                                             0.2s
 ✔ Network repo_default Removed                                             0.2s


In [107]:
!docker ps

CONTAINER ID   IMAGE      COMMAND                  CREATED          STATUS          PORTS                                     NAMES
60961ec617fc   repo-web   "/docker-entrypoint.…"   11 seconds ago   Up 11 seconds   0.0.0.0:8080->80/tcp, [::]:8080->80/tcp   repo-web-1


In [12]:
%%writefile .dockerignore
.Trash
.ipynb_checkpoints
__pycache__
*.pyc
.env
.git
.gitignore
.DS_Store
README.md
Untitled3.ipynb
node_modules

Overwriting .dockerignore


In [17]:
%%writefile main.py

import os
from pathlib import Path
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles

# 1️⃣ Создаём FastAPI
app = FastAPI()

# 2️⃣ Монтируем папку demo как статику
app.mount(
    "/",
    StaticFiles(directory="demo", html=True),
    name="static"
)

# 3️⃣ Вот сюда вставляется startup_event
@app.on_event("startup")
def startup_event():
    print("🚀 Startup event triggered")

    Path("demo").mkdir(exist_ok=True)

    Path("demo/index.html").write_text(
        "<h1>TEST PAGE WORKS</h1>",
        encoding="utf-8"
    )

    print("✅ Test file created")

Overwriting main.py
